In [1]:
import os, sys, json, math, time, re, random, hashlib, inspect
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List

import numpy as np
from collections import Counter

import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))


Torch: 2.10.0+cu128
CUDA: True | GPU count: 2
0 Tesla T4
1 Tesla T4


In [ ]:
# ============================================================
# 1. Config
# ============================================================
def first_existing(*paths) -> Path:
    for p in map(Path, paths):
        if p.exists():
            return p
    raise FileNotFoundError("Không tìm thấy path nào: " + " | ".join(map(str, paths)))

DATA_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/dataset-math",
    "/kaggle/input/dataset-math",
)
MODEL_NAME = str(first_existing(
    "/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/gpt2-vietnamese",
))

TRAIN_FILE = DATA_DIR / "train.json"
VALID_FILE = DATA_DIR / "valid.json"

# PROMPT_TEMPLATE = "Câu hỏi: {q}\nLời giải:\n"

PROMPT_TEMPLATE = "Câu hỏi: {q}\nLời giải từng bước:\n"

SAFE_EOS_ID = 50256

# Fast-Dev Mode Toggle
FAST_DEV = True  # flip to False for full Kaggle run
MAX_TRAIN_SAMPLES = 40000 if FAST_DEV else None
MAX_VALID_SAMPLES = 500 if FAST_DEV else None

OUTPUT_DIR = Path("/kaggle/working/gpt2_math_ckpt")
VALID_OUTPUT_PATH = Path("/kaggle/working/valid_output.json")
VALID_REPORT_PATH = Path("/kaggle/working/valid_report.json")


EPOCHS = 3
LR = 7e-5  
WARMUP_RATIO = 0.06 
MAX_LENGTH = 384  # Reduced from 512 for memory/speed based on 350 tok limit
PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM = 4
SEED = 42
WEIGHT_DECAY = 0.02

MAX_NEW_TOKENS = 150 # Reduced from 256 for inference efficiency
RUN_BASELINE_FIRST = False

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

print("TRAIN_FILE:", TRAIN_FILE)
print("VALID_FILE:", VALID_FILE)
print("MODEL_NAME:", MODEL_NAME)


TRAIN_FILE: /kaggle/input/datasets/kimanh2002/dataset-math/train.json
VALID_FILE: /kaggle/input/datasets/kimanh2002/dataset-math/valid.json
MODEL_NAME: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese


In [3]:
# Quick check: model must load offline
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, local_files_only=True)

tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID
model.config.pad_token_id = SAFE_EOS_ID
model.config.eos_token_id = SAFE_EOS_ID

print(type(tokenizer))
print(type(model))
print("vocab_size:", model.config.vocab_size)

del model
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<class 'transformers.models.gpt2.tokenization_gpt2.GPT2Tokenizer'>
<class 'transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel'>
vocab_size: 50257


# Training Pipeline

## Evaluation utilities

In [ ]:
# ============================================================
# 1.5 Evaluation utilities
# ============================================================
# Put the new strict anchor at the very top of the list
VI_ANCHORS = [
    r"(?i)đáp án là\s*[:：]?\s*(-?\d+(?:[.,]\d+)?)",
    r"(?i)câu trả lời là\s*[:：]?\s*(-?\d+(?:[.,]\d+)?)",
    
    # 2. LEGACY ANCHORS: Fallback cắt phần đuôi nếu Precision thất bại
    r"####đáp án là:\s*",       
    r"####\s*",                 
    r"Đáp án là\s*[:：]?",       
    r"Câu trả lời là\s*[:：]?",  
]

EN_ANCHORS = [
    r"(?i)the answer is\s*[:：]?\s*(-?\d+(?:[.,]\d+)?)",
    r"The answer is\s*[:：]?",
    r"####",
]

BOXED_RE = re.compile(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")
SAFE_NS = {"sqrt": math.sqrt, "pi": math.pi}


def extract_answer(text: str | None, anchors: list[str]) -> str | None:
    """
    Extract final answer by anchors. 
    - If anchor has a capture group (...), returns the exact group (Precision Mode).
    - Otherwise, cuts text after anchor and returns the tail (Legacy Mode).
    - Fallbacks to \boxed{}.
    """
    if not text:
        return None

    for anc in anchors:
        m = re.search(anc, text)
        if m:
            # TÍCH HỢP PRECISION: Kiểm tra xem Regex có capture group () không
            if m.lastindex: 
                return m.group(1).strip() # Lấy chính xác giá trị số trong ngoặc
            
            # LEGACY FALLBACK: Cắt toàn bộ phần đuôi
            tail = text[m.end():].strip()
            tail = tail.split("\n")[0]
            return tail.strip().rstrip(".。、,")

    # Fallback to LaTeX box
    boxes = BOXED_RE.findall(text)
    if boxes:
        return boxes[-1].strip()

    return None


def extract_gold(rec: dict) -> str | None:
    """Gold answer is read only from gold records, not from prediction files."""
    return extract_answer(rec.get("response_vi"), VI_ANCHORS)


def extract_pred(rec: dict) -> str | None:
    """
    Version 1 with defensive numeric cleanup.
    Works correctly when truncate_at_answer() is applied upstream.
    """
    text = rec.get("model_output", "")
    if not text:
        return None

    # Primary: precision anchor extraction
    ans = extract_answer(text, VI_ANCHORS + EN_ANCHORS)

    if ans is not None:
        # Defensive cleanup: if legacy anchor fired and returned tail text,
        # extract just the first number from it
        # If precision anchor fired, ans is already a pure number — this is a no-op
        num_match = re.search(r'-?\d+(?:[.,]\d+)?', ans)
        if num_match:
            return num_match.group(0)
        # Anchor found but no number — genuine failure
        return None

    # Last resort: the anchor was somehow absent (should not happen after truncation)
    # Return last number in text rather than None — score=1 is better than score=0
    nums = re.findall(r'-?\d+(?:[.,]\d+)?', text)
    return nums[-1] if nums else None
    

def parse_number(s: str | None) -> float | None:
    """Best-effort parser for finite scalar numeric answers."""
    if s is None:
        return None

    t = s.strip()
    if not t:
        return None

    # Pure Vietnamese decimal: 1,5 -> 1.5
    if re.fullmatch(r"-?\d+,\d+", t):
        try:
            return float(t.replace(",", "."))
        except ValueError:
            return None

    # Pure English decimal / integer
    if re.fullmatch(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?", t):
        try:
            val = float(t)
            return val if math.isfinite(val) else None
        except ValueError:
            return None

    # Strip variable assignment: x = 5 -> 5
    m = re.match(r"^[A-Za-z_]\w*\s*=\s*(.+)$", t)
    if m:
        t = m.group(1).strip()

    # Reject tuples / intervals early
    if t.startswith("(") and t.endswith(")") and re.search(r"\d\s*,\s*\d", t):
        return None
    if t.startswith("[") and t.endswith("]"):
        return None

    # Strip wrappers
    for _ in range(3):
        new = re.sub(r"\\boxed\{((?:[^{}]|\{[^{}]*\})*)\}", r"(\1)", t)
        if new == t:
            break
        t = new

    t = re.sub(r"\\text\{[^}]*\}", "", t)
    t = re.sub(r"\\mathrm\{[^}]*\}", "", t)
    t = t.replace("$", "")

    for token in ("\\,", "\\!", "\\;", "\\ ", "\\left", "\\right"):
        t = t.replace(token, "")

    for token in ("\\cdot", "\\times"):
        t = t.replace(token, "*")

    # LaTeX fractions / roots
    t = re.sub(r"\\(?:d|t)?frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"((\1)/(\2))", t)
    t = re.sub(r"\\sqrt\s*\{([^{}]+)\}", r"sqrt(\1)", t)
    t = re.sub(r"\\sqrt\s*(\d+(?:\.\d+)?)", r"sqrt(\1)", t)
    t = t.replace("\\pi", "pi")

# ── AST Compiler Protection (Fix for Float SyntaxWarning) ──────────
    # Implicit multiplication hardened for decimals and bracket-to-bracket
    t = re.sub(r"([\d.])\s*(sqrt|pi|\()", r"\1*\2", t)
    t = re.sub(r"(\))\s*(sqrt|pi|[\d.])", r"\1*\2", t)
    t = re.sub(r"(pi)\s*(sqrt|pi|[\d.]|\()", r"\1*\2", t)
    t = re.sub(r"\)\s*\(", r")*(", t) # <--- CRITICAL FIX FOR (1.5)(2.5)
    # ───────────────────────────────────────────────────────────────────

    # Comma handling
    has_period = "." in t
    n_commas = t.count(",")

    if n_commas == 1 and not has_period and re.search(r"\d,\d", t):
        # Vietnamese decimal
        t = re.sub(r"(?<=\d),(?=\d)", ".", t)
    elif n_commas >= 1:
        # English thousands separator: 1,000 -> 1000
        t = re.sub(r"(?<=\d),(?=\d{3}\b)", "", t)

    t = re.sub(r"\s+", "", t)

    if not t:
        return None

    # Reject remaining tuples/lists
    if "," in t:
        return None

    # Only allow safe expression chars
    leftover = re.sub(r"sqrt|pi|\d|\.|\+|\-|\*|/|\(|\)|\^|e|E", "", t)
    if leftover:
        return None

    t = t.replace("^", "**")

    try:
        val = eval(t, {"__builtins__": {}}, SAFE_NS)
    except Exception:
        return None

    if isinstance(val, bool):
        return None

    if isinstance(val, (int, float)):
        val = float(val)
        return val if math.isfinite(val) else None

    return None


def rel_error(pred: float | None, gold: float | None) -> float | None:
    if pred is None or gold is None:
        return None

    denom = max(1.0, abs(gold))
    return abs(pred - gold) / denom


def score_one(re_val: float | None, extractable: bool) -> int:
    if not extractable:
        return 0
    if re_val is None:
        return 0
    if re_val <= 0.01:
        return 10
    if re_val <= 0.10:
        return 5
    if re_val <= 0.50:
        return 1
    return 0


def align_predictions_with_gold(pred_items: list[dict], gold_items: list[dict]) -> list[tuple[dict, dict]]:
    """
    Align predictions and gold records.

    If both sides have id, align by id.
    Otherwise, require the same length and align by order.
    """
    pred_has_id = all("id" in x for x in pred_items)
    gold_has_id = all("id" in x for x in gold_items)

    if pred_has_id and gold_has_id:
        pred_map = {str(x["id"]): x for x in pred_items}
        pairs = []

        missing = []
        for g in gold_items:
            gid = str(g["id"])
            if gid not in pred_map:
                missing.append(gid)
            else:
                pairs.append((pred_map[gid], g))

        if missing:
            raise ValueError(f"Prediction thiếu {len(missing)} id, ví dụ: {missing[:5]}")

        return pairs

    if len(pred_items) != len(gold_items):
        raise ValueError(
            f"Số lượng prediction ({len(pred_items)}) khác số lượng gold ({len(gold_items)})."
        )

    return list(zip(pred_items, gold_items))


def evaluate(pred_items: list[dict], gold_items: list[dict]) -> dict:
    pairs = align_predictions_with_gold(pred_items, gold_items)

    rows = []
    total = 0
    bucket10 = bucket5 = bucket1 = bucket0 = 0
    extractable = 0
    numeric_pairs = 0
    rel_errors = []

    for pred_rec, gold_rec in pairs:
        gold_ans = extract_gold(gold_rec)
        pred_ans = extract_pred(pred_rec)
        # pred_ans = extract_pred_with_sanity(pred_rec)

        is_extractable = pred_ans is not None
        extractable += int(is_extractable)

        gold_num = parse_number(gold_ans)
        pred_num = parse_number(pred_ans)
        re_val = rel_error(pred_num, gold_num)

        if gold_num is not None and pred_num is not None and re_val is not None:
            numeric_pairs += 1
            rel_errors.append(re_val)

        s = score_one(re_val, is_extractable)
        total += s

        bucket10 += int(s == 10)
        bucket5 += int(s == 5)
        bucket1 += int(s == 1)
        bucket0 += int(s == 0)

        rows.append({
            "id": gold_rec.get("id", pred_rec.get("id")),
            "type": gold_rec.get("type") or pred_rec.get("type"),
            "gold_answer": gold_ans,
            "pred_answer": pred_ans,
            "gold_num": gold_num,
            "pred_num": pred_num,
            "rel_error": re_val,
            "extractable": is_extractable,
            "score": s,
        })

    n = len(rows)
    max_score = n * 10
    score_10 = total / n if n else 0.0

    return {
        "summary": {
            "n": n,
            "raw_score": total,
            "max_raw_score": max_score,
            "score_10": score_10,
            "score_pct": total / max_score if max_score else 0.0,
            "extractable": extractable,
            "numeric_pairs": numeric_pairs,
            "buckets": {
                "10": bucket10,
                "5": bucket5,
                "1": bucket1,
                "0": bucket0,
            },
            "rel_error_mean": sum(rel_errors) / len(rel_errors) if rel_errors else None,
        },
        "rows": rows,
    }

def save_evaluation_report(
    pred_path: str | Path,
    gold_records: list[dict],
    report_path: str | Path,
) -> dict:
    pred_path = Path(pred_path)
    report_path = Path(report_path)

    with pred_path.open("r", encoding="utf-8") as f:
        pred_items = json.load(f)

    result = evaluate(pred_items, gold_records)

    print(json.dumps(result["summary"], ensure_ascii=False, indent=2))

    with report_path.open("w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    print(f"Wrote {report_path}")
    return result

## Data loading & Preprocessing

**Chuyển các từ chỉ số và các từ chỉ phép toán thành số/ký hiệu toán học thực sự**

In [7]:
import re
from functools import lru_cache

# ══════════════════════════════════════════════════════════════════════════
# SECTION 1 — Core lookup tables
# ══════════════════════════════════════════════════════════════════════════

_SIMPLE: dict[str, int] = {
    "không": 0,
    "một": 1,  "mốt": 1,
    "hai": 2,
    "ba": 3,
    "bốn": 4,  "tư": 4,
    "năm": 5,  "lăm": 5,
    "sáu": 6,
    "bảy": 7,  "bay": 7,
    "tám": 8,
    "chín": 9,
    "mười": 10, "muời": 10,
}

_MAG: dict[str, int] = {
    "mươi":  10,
    "mười":  10,
    "trăm":  100,
    "nghìn": 1_000, "ngàn": 1_000,
    "triệu": 1_000_000,
    "tỷ":    1_000_000_000, "tỉ": 1_000_000_000,
}

_ALL_WORDS: list[str] = sorted(
    list(_SIMPLE.keys()) + list(_MAG.keys()),
    key=len, reverse=True
)

_NUM_WORD_ALT: str = "|".join(re.escape(w) for w in _ALL_WORDS)
_DIGIT_OR_WORD: str = rf"(?:\d+(?:[.,]\d+)?|{_NUM_WORD_ALT})"


# ══════════════════════════════════════════════════════════════════════════
# SECTION 2 — Guard tables
# ══════════════════════════════════════════════════════════════════════════

# FIX-4: added "lần" (ordinal counter: "lần đầu" = first time)
_ORDINAL_GUARDS: frozenset[str] = frozenset([
    "thứ",    # thứ hai = second, thứ ba = third
    "tháng",  # tháng một = January
    "ngày",   # ngày một = day 1 (calendar)
    "tuần",   # tuần một = week 1
    "lần",    # lần một = first time  ← NEW
])

# FIX-1: temporal unit set — if previous output token ends with a digit,
# these words denote duration/time, NOT a numeric value.
_TEMPORAL_UNITS: frozenset[str] = frozenset([
    "năm",    # years  (also = 5, hence the ambiguity)
    "tháng",  # months (also guarded above as ordinal)
    "tuần",   # weeks
    "ngày",   # days
    "giờ",    # hours
    "phút",   # minutes
    "giây",   # seconds
])

# FIX-2: semantic pairs where a classifier word precedes a number word
# and the combination should NOT be digit-converted.
# { prefix_word: {protected_number_words} }
_SEMANTIC_PROTECTED_PAIRS: dict[str, frozenset[str]] = {
    "bậc":  frozenset(["một", "hai", "ba", "bốn", "năm"]),   # bậc hai = squared
    "loại": frozenset(["một", "hai", "ba", "bốn"]),           # loại một = category 1
    "hạng": frozenset(["một", "hai", "ba", "bốn"]),  
    "phần": frozenset(["trăm"]),                             # FIX: Protect phần trăm# hạng nhất/hai
}


# ══════════════════════════════════════════════════════════════════════════
# SECTION 3 — Compound number parser  (unchanged from original)
# ══════════════════════════════════════════════════════════════════════════

def _parse_compound_number(tokens: list[str]) -> tuple[int | None, int]:
    """
    Greedily parse a Vietnamese compound number from the token list.
    Returns (value, tokens_consumed) or (None, 0).
    """
    if not tokens:
        return None, 0

    t = tokens[0].lower()
    if t not in _SIMPLE and t not in _MAG:
        return None, 0

    if t in _MAG and t not in _SIMPLE:
        mag = _MAG[t]
        consumed = 1
        sub_val, sub_consumed = _parse_compound_number(tokens[consumed:])
        if sub_val is not None and sub_val < mag:
            return mag + sub_val, consumed + sub_consumed
        return mag, consumed

    lead = _SIMPLE[t]
    consumed = 1

    if consumed >= len(tokens):
        return lead, consumed

    nxt = tokens[consumed].lower()

    if nxt in ("mươi", "mười") and lead >= 2:
        consumed += 1
        base = lead * 10
        if consumed < len(tokens):
            nnxt = tokens[consumed].lower()
            if nnxt in _SIMPLE and nnxt not in ("mươi", "mười"):
                base += _SIMPLE[nnxt]
                consumed += 1
        return base, consumed

    if nxt == "mười" and lead == 1:
        consumed += 1
        base = 10
        if consumed < len(tokens):
            nnxt = tokens[consumed].lower()
            if nnxt in _SIMPLE and nnxt not in ("mươi", "mười"):
                base += _SIMPLE[nnxt]
                consumed += 1
        return base, consumed

    if t in ("mười", "muời") and nxt in _SIMPLE and nxt not in ("mươi", "mười"):
        return 10 + _SIMPLE[nxt], 2

    if nxt == "trăm":
        consumed += 1
        base = lead * 100
        sub_val, sub_consumed = _parse_compound_number(tokens[consumed:])
        if sub_val is not None and sub_val < 100:
            return base + sub_val, consumed + sub_consumed
        return base, consumed

    if nxt in ("nghìn", "ngàn"):
        consumed += 1
        base = lead * 1_000
        sub_val, sub_consumed = _parse_compound_number(tokens[consumed:])
        if sub_val is not None and sub_val < 1_000:
            return base + sub_val, consumed + sub_consumed
        return base, consumed

    if nxt == "triệu":
        consumed += 1
        base = lead * 1_000_000
        sub_val, sub_consumed = _parse_compound_number(tokens[consumed:])
        if sub_val is not None and sub_val < 1_000_000:
            return base + sub_val, consumed + sub_consumed
        return base, consumed

    if nxt in ("tỷ", "tỉ"):
        consumed += 1
        return lead * 1_000_000_000, consumed

    return lead, 1


# ══════════════════════════════════════════════════════════════════════════
# SECTION 4 — Public function 1: number-word normalizer
# ══════════════════════════════════════════════════════════════════════════

def normalize_vietnamese_number_words(text: str) -> str:
    """
    Replace Vietnamese number-word sequences with digit equivalents
    in mathematical contexts.

    Safety guarantees
    -----------------
    • Ordinal prefixes protect following number words.
    • FIX-1: Temporal units ("năm", "ngày", …) are NOT converted when
      immediately preceded by a digit token.  "5 năm" stays "5 năm".
    • FIX-2: Semantic pairs ("bậc hai", "loại ba") are NOT converted.
    • "nửa" (half) → "0.5" only in arithmetic context.
    • Idempotent.
    """
    if not isinstance(text, str):
        return text

    tokens = text.split()
    out: list[str] = []
    i = 0

    while i < len(tokens):
        tok_lower = tokens[i].lower()

        # ── Ordinal / calendar guard ──────────────────────────────────
        if tok_lower in _ORDINAL_GUARDS:
            out.append(tokens[i])
            i += 1
            if i < len(tokens) and tokens[i].lower() in _SIMPLE:
                out.append(tokens[i])
                i += 1
            continue

        # ── FIX-1: Temporal unit protection ──────────────────────────
        # "5 năm" must stay "5 năm", not "5 5"
        if tok_lower in _TEMPORAL_UNITS:
            prev_out = out[-1] if out else ""
            if re.search(r"\d$", prev_out):
                # Previous token ended with a digit → this is a duration word
                out.append(tokens[i])
                i += 1
                continue
            # No preceding digit → could still be a standalone number word
            # (e.g. sentence-initial "năm học sinh"), fall through to parser

        # ── FIX-2: Semantic phrase protection ────────────────────────
        # "bậc hai" → keep "hai" as-is; "phần trăm" → keep "trăm" as-is
        if tok_lower in _SIMPLE or tok_lower in _MAG:
            prev_out = out[-1].lower().strip(".,;:!?") if out else ""
            if (prev_out in _SEMANTIC_PROTECTED_PAIRS and
                    tok_lower in _SEMANTIC_PROTECTED_PAIRS[prev_out]):
                out.append(tokens[i])
                i += 1
                continue
        # ── "nửa" (half) → 0.5 only in arithmetic context ────────────
        if tok_lower == "nửa":
            prev_is_num = bool(out and re.search(r"\d", out[-1]))
            next_is_num = (i + 1 < len(tokens) and
                           re.search(r"\d|[+\-*/]", tokens[i + 1]))
            out.append("0.5" if (prev_is_num or next_is_num) else tokens[i])
            i += 1
            continue

        # ── Try compound number parse ─────────────────────────────────
        # KHÔI PHỤC: Gọi hàm phân tích số bị thiếu
        if tok_lower in _SIMPLE or tok_lower in _MAG:
            val, consumed = _parse_compound_number(tokens[i:])
            if val is not None and consumed > 0:
                out.append(str(val))
                i += consumed
                continue
                
        out.append(tokens[i])
        i += 1

    return " ".join(out)


# ══════════════════════════════════════════════════════════════════════════
# SECTION 5 — Public function 2: math-operator normalizer
# ══════════════════════════════════════════════════════════════════════════

_NB = rf"(?:{_DIGIT_OR_WORD})"

_NHAN_NON_ARITH_FOLLOW: re.Pattern = re.compile(
    r"^(viên|vật|dân|loại|tố|lực|chứng|quyền|giống|tạo|bản|từ|điển|ái|đức|nghĩa|hậu)\b",
    re.IGNORECASE,
)

_CHIA_NON_ARITH_FOLLOW: re.Pattern = re.compile(
    r"^(sẻ|tay|rẽ|cắt|lửa|buồn|vui|ngọt)\b",
    re.IGNORECASE,
)

_FRACTION_RE: re.Pattern = re.compile(
    rf"({_NB})\s+phần\s+({_NB})",
    re.IGNORECASE,
)

_PCT_RE: re.Pattern = re.compile(
    rf"({_NB})\s+phần\s+trăm",
    re.IGNORECASE,
)

# FIX-3: standalone "phần N" (already digit after word-normalizer ran)
# "phần 3" → "1/3",  "phần 10" → "1/10"
# Applied AFTER normalize_vietnamese_number_words so denominators are digits.
_STANDALONE_FRAC_DIGIT_RE: re.Pattern = re.compile(
    r"\bphần\s+(\d+)\b"
)

_NHAN_VOI_RE: re.Pattern = re.compile(r"(?<=\d\s)nhân\s+với(?=\s)", re.IGNORECASE)
_CHIA_CHO_RE: re.Pattern = re.compile(r"(?<=\d\s)chia\s+cho(?=\s)", re.IGNORECASE)
_CONG_VOI_RE: re.Pattern = re.compile(r"(?<=\d\s)cộng\s+với(?=\s)", re.IGNORECASE)
_TRU_DI_RE:  re.Pattern = re.compile(r"(?<=\d\s)trừ\s+đi(?=\s)", re.IGNORECASE)

_GAP_RE: re.Pattern = re.compile(
    r"\bgấp\s+(\d+(?:[.,]\d+)?|" + _NUM_WORD_ALT + r")(?:\s+lần)?",
    re.IGNORECASE,
)


def _is_fraction_context(m: re.Match, text: str) -> bool:
    numerator_str = m.group(1)
    return bool(re.search(r"\d", numerator_str) or
                numerator_str.lower() in _SIMPLE)


def normalize_vietnamese_math_operators(text: str, *,
                                        convert_pct: bool = True,
                                        convert_gap: bool = True) -> str:
    """
    Replace Vietnamese operator words with arithmetic symbols.

    Additions vs. original
    ----------------------
    FIX-3: Step 1.5 — standalone "phần N" → "1/N" (runs after word→digit pass)
    """
    if not isinstance(text, str):
        return text

    # Step 0: word→digit (idempotent)
    text = normalize_vietnamese_number_words(text)

    # Step 1: percent  "N phần trăm" → "N%"
    if convert_pct:
        text = _PCT_RE.sub(lambda m: m.group(1) + "%", text)

    # Step 2 MUST run before Step 3 to prevent "1 phần 3" from turning into "1 1/3".
    prev = None
    while prev != text:
        prev = text
        text = _FRACTION_RE.sub(
            lambda m: (f"{m.group(1)}/{m.group(2)}"
                       if _is_fraction_context(m, text)
                       else m.group(0)),
            text,
        )

    # Step 3: standalone fraction  "phần 3" → "1/3"
    text = _STANDALONE_FRAC_DIGIT_RE.sub(lambda m: f"1/{m.group(1)}", text)

    # Step 4: verb-phrase operators (unambiguous multi-word)
    text = _NHAN_VOI_RE.sub("*", text)
    text = _CHIA_CHO_RE.sub("/", text)
    text = _CONG_VOI_RE.sub("+", text)
    text = _TRU_DI_RE.sub("-", text)

    #Step 5: "gấp N [lần]" → "* N"
    if convert_gap:
        def _gap_repl(m: re.Match) -> str:
            raw_n = m.group(1)
            digit_n = normalize_vietnamese_number_words(raw_n)
            return f"* {digit_n}"
        text = _GAP_RE.sub(_gap_repl, text)

    # Step 6: context-guarded single-word operators
    tokens = text.split(" ")
    out: list[str] = []
    
    for idx, tok in enumerate(tokens):
        tok_l = tok.lower()

        if tok_l == "cộng":
            if _numeric_neighbour(tokens, idx):
                out.append("+"); continue

        elif tok_l == "trừ":
            next_tok = tokens[idx + 1].lower() if idx + 1 < len(tokens) else ""
            if next_tok not in ("phi", "khi") and _numeric_neighbour(tokens, idx):
                out.append("-"); continue

        elif tok_l == "nhân":
            next_tok = tokens[idx + 1] if idx + 1 < len(tokens) else ""
            if not _NHAN_NON_ARITH_FOLLOW.match(next_tok) and _numeric_neighbour(tokens, idx):
                out.append("*"); continue

        elif tok_l == "chia":
            next_tok = tokens[idx + 1] if idx + 1 < len(tokens) else ""
            if not _CHIA_NON_ARITH_FOLLOW.match(next_tok) and _numeric_neighbour(tokens, idx):
                out.append("/"); continue

        out.append(tok)

    return " ".join(out)

def _numeric_neighbour(tokens: list[str], idx: int, window: int = 1) -> bool:
    _num_word_set = set(_SIMPLE.keys()) | set(_MAG.keys())
    for delta in range(1, window + 2):
        for sign in (-1, 1):
            j = idx + sign * delta
            if 0 <= j < len(tokens):
                t = tokens[j].lower().strip(".,;:!?()")
                if re.search(r"\d", t) or t in _num_word_set:
                    return True
    return False

# ══════════════════════════════════════════════════════════════════════════
# SECTION 6 — Combined convenience wrapper
# ══════════════════════════════════════════════════════════════════════════
import unicodedata

def normalize_vi_math_text(text: str) -> str:
    """
    Apply both normalisers in the correct order.
    Call this BEFORE clean_math_text() in the training pipeline.
    """
    # Normalize to NFC before any Vietnamese word matching
    text = unicodedata.normalize('NFC', text)
    text = normalize_vietnamese_number_words(text)
    text = normalize_vietnamese_math_operators(text)
    return text

In [8]:
# Check
assert normalize_vi_math_text("tám ô") == "8 ô", "NORMALIZATION BUG: Unicode mismatch"
assert normalize_vi_math_text("năm học sinh") == "5 học sinh", "Temporal guard broken"  
assert normalize_vi_math_text("5 năm") == "5 năm", "Should not convert temporal năm"

test_pred = {"model_output": "Vậy cô ấy cần 72 ô nữa.\n####đáp án là: 72 ô"}
ans = extract_pred(test_pred)
assert ans == "72", f"Extraction bug: got '{ans}', expected '72'"
print("[PASS] All pipeline health checks passed")

[PASS] All pipeline health checks passed


In [ ]:
# ============================================================
# 2. Data loading & Preprocessing
# ============================================================
def load_records(path: str | Path) -> list:
    p = Path(path)
    with p.open("r", encoding="utf-8") as f:
        head = f.read(1)
        f.seek(0)
        records = json.load(f) if head == "[" else [json.loads(line) for line in f if line.strip()]
    # Normalize all string fields to NFC at load time
    for rec in records:
        for key in rec:
            if isinstance(rec[key], str):
                rec[key] = unicodedata.normalize('NFC', rec[key])
    return records
    
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_dir(dir_path: Path, suffixes=(".bin", ".safetensors", ".json", ".txt", ".model")) -> str:
    h = hashlib.sha256()
    for p in sorted(x for x in dir_path.rglob("*") if x.is_file() and x.suffix in suffixes):
        h.update(p.relative_to(dir_path).as_posix().encode() + b"\0")
        h.update(sha256_file(p).encode() + b"\0")
    return h.hexdigest()

# ============================================================
# Text & Typographical Cleansing
# ============================================================
def clean_math_text(text: str) -> str:
    if not isinstance(text, str): return text

    # CRITICAL: Protect <<expr=result>> annotations BEFORE any substitutions
    # Extract and temporarily replace them
    placeholders = {}
    def protect_annotation(m):
        key = f"__ANNOT_{len(placeholders)}__"
        placeholders[key] = m.group(0)
        return key
    
    text = re.sub(r'<<[^>]+>>', protect_annotation, text)
    
    # 1. LaTeX cleanup
    text = re.sub(r'\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}', r'(\1/\2)', text)
    text = re.sub(r'\\sqrt\s*\{([^{}]+)\}', r'sqrt(\1)', text)
    text = text.replace('\\times', ' * ').replace('\\cdot', ' * ')
    text = re.sub(r'\\angle\s*([A-Z]{2,3})', r'góc \1', text)
    text = re.sub(r'\\triangle\s*([A-Z]{2,3})', r'tam giác \1', text)
    text = re.sub(r'\\[a-zA-Z]+\{([^{}]*)\}', r'\1', text)
    text = re.sub(r'\\[a-zA-Z]+\s', ' ', text)
    text = re.sub(r'\$([^$]{1,100})\$', r'\1', text)
    text = re.sub(r'\$\$([^$]+)\$\$', r'\1', text)
    
    # 2. Fix digit * digit corruptions
    text = re.sub(r'((?<=\d)\s*_\s*(?=\d))', ' * ', text)
    text = re.sub(r'([A-Za-z])_(\d+)', r'\1\2', text)
    text = re.sub(r'\^\{(\d+)\}', r'^\1', text)
    
    # 3. Numeric normalizations
    text = re.sub(r'(?<=\d),(?=\d{3}\b)', '', text)
    text = re.sub(r'(?<=\d)\.(?=\d{3}\b)', '', text)
    text = re.sub(r'(?<=\d),(?=\d)', '.', text)
    
    # Restore protected annotations
    for key, val in placeholders.items():
        text = text.replace(key, val)
    
    return re.sub(r'\s+', ' ', text).strip()

# ============================================================
# Format Normalization & Strict Numeric Sanitization
# ============================================================
def normalize_response(response: str) -> str | None:
    gold_str = None
    
    # 1. Quét tìm số liệu đích bằng các patterns từ EDA
    # Lưu ý: Tìm `đáp án là:` trước, nếu không có mới tìm `####`
    patterns = [
        r"(?i)(?:đáp án|câu trả lời)\s*là\s*[:：]?\s*(-?\d+(?:[.,]\d+)?)",
        r"####\s*(-?\d+(?:[.,]\d+)?)" # Cứu cánh cho các mẫu bị lỗi như "####10 đáp án là:"
    ]
    
    for pat in patterns:
        matches = re.findall(pat, response)
        if matches:
            gold_str = matches[-1].replace(',', '.') # Luôn lấy số xuất hiện cuối cùng
            break
            
    if not gold_str:
        return None
        
    gold_num = parse_number(gold_str)
    if gold_num is None or isinstance(gold_num, bool):
        return None
        
    # 2. Chặt đứt TOÀN BỘ rác ở đuôi (bao gồm cả "####5 đáp án là:")
    # Regex split này sẽ cắt text ngay tại ký tự '#' hoặc chữ 'đáp' đầu tiên nó gặp ở phần kết luận.
    reasoning = re.split(r'(?i)####|đáp án là|câu trả lời là', response)[0].strip()
    
    # 3. Tái tạo lại (Reconstruct) với Template tinh khiết
    clean_num = str(int(gold_num)) if gold_num == int(gold_num) else str(gold_num)
    return f"{reasoning}\n####đáp án là: {clean_num}"

def is_short_cot(rec: dict, max_steps: int = 5) -> bool:
    """
    Giữ lại các sample có độ dài chuỗi suy luận (CoT) <= max_steps.
    """
    target = rec.get('_target', '')
    step_pattern = re.compile(r'[\d\.\,]+\s*[\+\-\*\/]\s*[\d\s\+\-\*\/\(\)\.\,]*\s*=\s*-?[\d\.\,]+')
    steps = step_pattern.findall(target)
    # Loại bỏ các bài không có phép tính (<= 0) và quá dài (> 4)
    return len(steps) <= max_steps

def is_concise_response(rec: dict, tokenizer, max_response_tokens: int = 150) -> bool:
    """
    Keep only responses where the FULL reasoning chain + anchor fits in 150 tokens.
    This is the TinyGSM key insight: short dense reasoning > long verbose reasoning.
    """
    target = rec.get('_target', '')
    r_ids = tokenizer(target, add_special_tokens=False)['input_ids']
    return len(r_ids) <= max_response_tokens
    

def compute_arithmetic_score(response_text: str) -> float:
    """
    Returns:
        1.0  — no verifiable arithmetic found (neutral, don't penalize)
        0.0–1.0 — fraction of verifiable expressions that are correct
    """
    response_text = clean_math_text(response_text)
    
    # Strict pattern: ONLY pure numeric expressions, no letters allowed nearby
    strict_pattern = re.compile(
        r'(?<![A-Za-zÀ-ỹ])'           # Not preceded by letter
        r'([\d\s\+\-\*\/\(\)\.]{3,})'  # Expression: at least 3 chars
        r'\s*=\s*'
        r'(-?[\d\.]+)'                  # Result
        r'(?![A-Za-zÀ-ỹ\d])'          # Not followed by letter or digit
    )
    matches = strict_pattern.findall(response_text)
    
    if not matches:
        return 1.0  # Neutral

    correct = total = 0
    for expr_str, result_str in matches:
        expr_str = expr_str.strip()
        # Final guard: reject if any letters slipped through
        if re.search(r'[A-Za-zÀ-ỹ]', expr_str):
            continue
        # Reject trivial "X = X" (just an assignment, not a computation)
        if re.fullmatch(r'\s*[\d\.]+\s*', expr_str):
            continue
        expr_str = re.sub(r'([\d.])\s*\(', r'\1*(', expr_str)
        expr_str = re.sub(r'\)\s*([\d.])', r')*\1', expr_str)
        expr_str = re.sub(r'\)\s*\(', r')*(', expr_str)
        try:
            computed = eval(expr_str, {"__builtins__": {}}, {})
            expected = float(result_str.strip())
            denom = max(1.0, abs(expected))
            if abs(float(computed) - expected) / denom < 0.02:
                correct += 1
            total += 1
        except Exception:
            pass  # Unevaluable → skip, don't penalize

    return (correct / total) if total > 0 else 1.0

def is_arithmetically_verified(rec: dict, threshold: float = 0.5) -> bool:
    """
    Filter: only keep training samples where at least 75% of 
    verifiable arithmetic statements in the response are correct.
    """
    return compute_arithmetic_score(rec.get('_target', '')) >= threshold

def is_final_answer_correct(rec: dict) -> bool:
    """Cross-validate: Đảm bảo output text thực sự chứa số khớp với gold."""
    gold_str = extract_answer(rec.get('response_vi', ''), VI_ANCHORS)
    parsed_num = parse_number(gold_str)
    return parsed_num is not None

def is_valid_length(rec, tokenizer, max_total_length=MAX_LENGTH, min_resp_len=20):
    prompt = PROMPT_TEMPLATE.format(q=rec['query_vi'].strip())
    p_len = len(tokenizer(prompt, add_special_tokens=False)['input_ids'])
    r_len = len(tokenizer(rec['_target'].strip(), add_special_tokens=False)['input_ids'])
    
    if (p_len + r_len + 1) > max_total_length:
        return False
    if r_len < min_resp_len:
        return False
    return True
    
def is_extractable_target(target_text: str) -> bool:
    """
    Giả lập lại quá trình chấm điểm của Kaggle pipeline.
    Kiểm tra xem target_text có trích xuất ra được một số hợp lệ không.
    Nếu không -> Toxic sample, phải loại bỏ.
    """
    extracted_str = extract_answer(target_text, VI_ANCHORS)
    
    if extracted_str is None:
        return False
        
    parsed_val = parse_number(extracted_str)
    
    # Nếu parse ra None, hoặc ra Boolean (lỗi logic), loại bỏ.
    if parsed_val is None or isinstance(parsed_val, bool):
        return False
        
    return True

# KEEP_MATH_TYPES = {'MATH_SV', 'MATH_FOBAR'}

# def is_learnable_sample(rec: dict) -> bool:
#     prob_type = rec.get('type', '')
#     if prob_type.startswith('MATH') and prob_type not in KEEP_MATH_TYPES:
#         return False
#     if len(rec.get('query_vi', '').split()) < 5:
#         return False
#     if '\\begin{array}' in rec.get('query_vi', ''):
#         return False
#     return True

In [10]:
# ============================================================
# Pipeline
# ============================================================
# 1. Load Raw Data
raw_train_records = load_records(TRAIN_FILE)
valid_records = load_records(VALID_FILE)

In [60]:
def eda_find_anchors(records: list[dict], field_name: str = 'response_vi', top_k: int = 20):
    """
    Quét tập dữ liệu để tìm ra các mẫu text (anchor) phổ biến nhất
    nằm ngay trước đáp án số cuối cùng.
    """
    anchor_counts = Counter()
    no_number_count = 0
    total_records = len(records)

    for rec in records:
        text = rec.get(field_name, '')
        if not text:
            continue

        # TÌm tất cả các số trong text (bao gồm số âm, số thập phân dấu chấm/phẩy)
        matches = list(re.finditer(r'-?\d+(?:[.,]\d+)?', text))

        if not matches:
            no_number_count += 1
            continue

        # Chú tâm vào con số cuối cùng (vì đây là target label)
        last_match = matches[-1]
        num_start_idx = last_match.start()

        # Lấy 30 ký tự ngay trước con số cuối cùng
        prefix = text[max(0, num_start_idx - 30):num_start_idx]

        # Làm sạch prefix để gom nhóm (Clustering):
        # 1. Chuyển thành chữ thường
        # 2. Lấy phần text sau dấu chấm, dấu phẩy, hoặc dấu xuống dòng cuối cùng để loại bỏ nhiễu lý thuyết
        clean_prefix = re.split(r'[\n\.,]', prefix)[-1]
        clean_prefix = clean_prefix.strip().lower()

        # Nếu string trống (số nằm ngay sau dấu xuống dòng/chấm), dán nhãn đặc biệt
        if not clean_prefix:
            clean_prefix = "<EMPTY_OR_NEWLINE>"

        anchor_counts[clean_prefix] += 1

    # In báo cáo chuyên nghiệp
    print(f"\n{'='*80}")
    print(f" BÁO CÁO EDA ANCHOR CHO TRƯỜNG: {field_name}")
    print(f"{'='*80}")
    print(f"* Tổng số samples quét: {total_records}")
    print(f"* Samples không tìm thấy số (Cần drop): {no_number_count}")
    
    total_valid = total_records - no_number_count
    
    # Tạo DataFrame để hiển thị đẹp trong Jupyter/Kaggle Notebook
    df_data = []
    for anchor, count in anchor_counts.most_common(top_k):
        pct = (count / total_valid) * 100 if total_valid > 0 else 0
        df_data.append([repr(anchor), count, f"{pct:.2f}%"])
        
    df = pd.DataFrame(df_data, columns=["Anchor_Pattern (Regex Literal)", "Count", "Percentage"])
    print("\nTOP 20 MỎ NEO PHỔ BIẾN NHẤT:")
    print(df.to_string(index=False))
    print(f"{'='*80}\n")

    return anchor_counts

In [62]:
vi_anchors_stats = eda_find_anchors(raw_train_records, field_name='response_vi')
en_anchors_stats = eda_find_anchors(raw_train_records, field_name='response_en')


 BÁO CÁO EDA ANCHOR CHO TRƯỜNG: response_vi
* Tổng số samples quét: 95400
* Samples không tìm thấy số (Cần drop): 41

TOP 20 MỎ NEO PHỔ BIẾN NHẤT:
Anchor_Pattern (Regex Literal)  Count Percentage
             'câu trả lời là:'  15464     16.22%
                  'đáp án là:'  13391     14.04%
           '####10 đáp án là:'   1287      1.35%
            '####5 đáp án là:'   1224      1.28%
            '####4 đáp án là:'   1218      1.28%
           '#### 3 đáp án là:'   1132      1.19%
            '####3 đáp án là:'   1125      1.18%
            '####2 đáp án là:'   1123      1.18%
            '####6 đáp án là:'   1110      1.16%
           '#### 2 đáp án là:'   1103      1.16%
           '####20 đáp án là:'   1042      1.09%
  'câu trả lời là: \\frac{1}{'    900      0.94%
           '#### 4 đáp án là:'    894      0.94%
           '####15 đáp án là:'    851      0.89%
           '####12 đáp án là:'    801      0.84%
           '#### 5 đáp án là:'    785      0.82%
          '#### 50 

In [11]:
# 2. Apply Fast-Dev limits early
if MAX_TRAIN_SAMPLES:
    raw_train_records = raw_train_records[:MAX_TRAIN_SAMPLES * 3] 
if MAX_VALID_SAMPLES:
    valid_records = valid_records[:MAX_VALID_SAMPLES]

print(f"Starting pipeline with {len(raw_train_records)} raw train records.")

# BƯỚC 3: Normalize text (TRƯỚC KHI set _target)
print("Applying word-to-digit normalization and typographical cleansing...")
for rec in raw_train_records:
    q_norm = normalize_vi_math_text(rec.get('query_vi', ''))
    r_norm = normalize_vi_math_text(rec.get('response_vi', ''))
    rec['query_vi']    = clean_math_text(q_norm)
    rec['response_vi'] = clean_math_text(r_norm)

for rec in valid_records:
    q_norm = normalize_vi_math_text(rec.get('query_vi', ''))
    r_norm = normalize_vi_math_text(rec.get('response_vi', ''))
    rec['query_vi']    = clean_math_text(q_norm)
    rec['response_vi'] = clean_math_text(r_norm)

# BƯỚC 5: SET _target — PHẢI TRƯỚC KHI GỌI is_short_cot
valid_normalized_count = 0
for rec in raw_train_records:
    normalized = normalize_response(rec['response_vi'])
    if normalized and is_extractable_target(normalized):
        rec['_target'] = normalized
        valid_normalized_count += 1

train_records = [r for r in raw_train_records if '_target' in r]
print(f"After normalize_response + is_extractable_target: {len(train_records)}")

# BƯỚC 6: is_short_cot — GIỜ MỚI ĐƯỢC GỌI (đọc _target đã tồn tại)
initial_len = len(train_records)
train_records = [r for r in train_records if is_short_cot(r, max_steps=5)]
print(f"After is_short_cot: {len(train_records)}/{initial_len} "
      f"({len(train_records)/max(1,initial_len)*100:.1f}%)")

# 4. Length Filtering (requires local tokenizer load - SLOW)
print(f"Loading tokenizer for length filtering (Max Total: {MAX_LENGTH})...")
temp_tok = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
temp_tok.pad_token_id = SAFE_EOS_ID 
temp_tok.eos_token_id = SAFE_EOS_ID

print("4. Deep filtering (Length, Correctness, Arithmetic)...")
train_records = [r for r in train_records if is_concise_response(r, temp_tok, max_response_tokens=150)]
print(f"   -> After conciseness check: {len(train_records)}")

train_records = [r for r in train_records if is_final_answer_correct(r)]
print(f"   -> After correctness check: {len(train_records)}")

train_records = [r for r in train_records if is_arithmetically_verified(r)]
print(f"   -> After arithmetic verification: {len(train_records)}")

train_records = [r for r in train_records if is_valid_length(r, temp_tok, max_total_length=MAX_LENGTH)]
print(f"   -> After total length check: {len(train_records)}")
del temp_tok

print(f"Original train subset (pre-filter): {len(raw_train_records)}")
print(f"Cleaned train (post-filter): {len(train_records)}")
print(f"Valid: {len(valid_records)}")

Starting pipeline with 95400 raw train records.
Applying word-to-digit normalization and typographical cleansing...
After normalize_response + is_extractable_target: 90604
After is_short_cot: 89448/90604 (98.7%)
Loading tokenizer for length filtering (Max Total: 384)...
4. Deep filtering (Length, Correctness, Arithmetic)...
   -> After conciseness check: 56658
   -> After correctness check: 56549
   -> After arithmetic verification: 52021
   -> After total length check: 51787
Original train subset (pre-filter): 95400
Cleaned train (post-filter): 51787
Valid: 500


In [21]:
# def curriculum_sort_key(rec: dict) -> tuple[int, int]:
#     """
#     Sắp xếp dữ liệu theo độ khó tuyến tính:
#     Ưu tiên 1: Số lượng phép tính (Đếm dấu =)
#     Ưu tiên 2: Độ dài chuỗi target (Đếm ký tự)
#     """
#     target = rec.get('_target', '')
    
#     # 1. Đếm số phép toán
#     steps = len(re.findall(r'=', target))
    
#     # 2. Chiều dài text
#     length = len(target)
    
#     return (steps, length)

# print("Sắp xếp Dataset theo Curriculum Learning (Easy -> Hard)...")
# train_records.sort(key=curriculum_sort_key)

Sắp xếp Dataset theo Curriculum Learning (Easy -> Hard)...


In [12]:
print(json.dumps(train_records[3], ensure_ascii=False, indent=2)[:2000])

{
  "original_question_vi": "Bart làm một mixtape. Mặt đầu tiên có 6 bài hát. Mặt thứ hai có 4 bài hát. Mỗi bài hát có thời lượng 4 phút. Tổng chiều dài băng là bao nhiêu?",
  "original_question_en": "Bart makes a mixtape.  The first side has 6 songs.  The second side has 4 songs.  Each song is 4 minutes.  How long is the total tape?",
  "query_vi": "Bart làm 1 mixtape. Mặt đầu tiên có 6 bài hát. Mặt thứ hai có x bài hát. Mỗi bài hát có thời lượng 4 phút. Tổng chiều dài băng là bao nhiêu? Nếu chúng ta biết câu trả lời cho câu hỏi trên là 40 thì giá trị của biến x chưa biết là bao nhiêu?",
  "query_en": "Bart makes a mixtape.  The first side has 6 songs.  The second side has x songs.  Each song is 4 minutes.  How long is the total tape?\nIf we know the answer to the above question is 40, what is the value of unknown variable x?",
  "response_vi": "Mặt đầu tiên của mixtape có 6 bài hát, mỗi bài có độ dài 4 phút nên tổng độ dài của mặt đầu tiên là 6 * 4 = 24 phút. Mặt thứ 2 của mixtape có

In [13]:
print(json.dumps(valid_records[2], ensure_ascii=False, indent=2)[:3000])

{
  "original_question_vi": "Một con súc sắc tám mặt có các mặt được đánh số từ 1 đến 8. Giá trị kỳ vọng của việc tung xúc xắc là bao nhiêu?",
  "original_question_en": "An eight-sided die has its faces numbered from 1 to 8. What is the expected value of the roll of the die?",
  "query_vi": "1 con súc sắc 8 mặt có các mặt được đánh số từ 1 đến X. Giá trị kỳ vọng của con xúc xắc là 4.5. Giá trị của biến X chưa biết là bao nhiêu?",
  "query_en": "An eight-sided die has its faces numbered from 1 to X. The expected value of the roll of the die is 4.5. What is the value of unknown variable X?",
  "response_vi": "Để giải bài toán này, chúng ta cần xác định giá trị của x, biểu thị số ở mặt cao nhất của xúc xắc 8 mặt. Giá trị kỳ vọng của việc tung xúc xắc được cho là 4.5. Điều này có nghĩa là trung bình, tổng các số trên tất cả các mặt chia cho số mặt là 4.5. Hãy thiết lập phương trình: (1 + 2 + 3 + 4 + 5 + 6 + 7 + x) / 8 = 4.5 Hãy đơn giản hóa và giải x: 28 + x = 36 Để cô lập x, chúng ta - 28

In [70]:
# import re

# def compute_cot_efficiency_score(rec: dict) -> float:
#     """
#     Evaluates the density of mathematical reasoning vs text verbosity.
#     """
#     target = rec.get('_target', rec.get('response_vi', ''))
#     words = target.split()
#     word_count = len(words)
    
#     # 1. Noise Filter: Reject extremely short/truncated responses
#     if word_count < 10: 
#         return 0.0
        
#     # 2. Math Density Calculation
#     # Improved regex: captures full numbers (decimals) and operators as single entities
#     ops_and_nums = len(re.findall(r'-?\d+(?:[.,]\d+)?|[\+\-\*\/=]', target))
#     density = ops_and_nums / word_count
    
#     # 3. Context Length Penalty (Adjusted for MAX_LENGTH = 384)
#     # Prompt is typically 50-100 tokens. Response safe zone is <= 150 words (~200 tokens).
#     length_penalty = 1.0
#     if word_count > 150:
#         # Gradual penalty starting at 150 words, bottoms out at 0.1
#         length_penalty = max(0.1, 1.0 - (word_count - 150) / 80.0)
        
#     return density * length_penalty

**What specific failure modes (e.g., LaTeX hallucinations, overly long queries, or specific problem types) are you trying to target with this filter?**

## SFT dataset and collator

In [ ]:
class SFTDataset(Dataset):
    def __init__(self, records, tokenizer, max_length: int):
        self.records = records
        self.tok = tokenizer
        self.max_length = max_length
        self.SAFE_EOS_ID = 50256

        # Pre-tokenize the anchor once for fast subsequence search
        self.stable_anchor_ids = self.tok("####", add_special_tokens=False)["input_ids"]
        self.stable_anchor_len = len(self.stable_anchor_ids)

        # Pre-build a set of token IDs that decode to math operator characters.
        # We check single-token decodings; multi-token operators are caught by context.
        self._eq_token_ids = self._find_operator_token_ids(['=', '+', '-', '*', '/'])

    def _find_operator_token_ids(self, operators: list[str]) -> set[int]:
        """
        Find all token IDs whose decoded string (stripped) is one of the operators.
        This handles cases where operators may be tokenized with surrounding spaces.
        """
        operator_ids = set()
        vocab_size = self.tok.vocab_size if hasattr(self.tok, 'vocab_size') else 50257
        # Only scan a representative range — operators are low-frequency tokens
        for token_id in range(vocab_size):
            try:
                decoded = self.tok.decode([token_id]).strip()
                if decoded in operators:
                    operator_ids.add(token_id)
            except Exception:
                continue
        return operator_ids

    def _build_arithmetic_weights(self, ids: List[int], p_ids_len: int) -> List[float]:
        """
        Fix D: Weight assignment strategy.
        
        1. Find all '=' token positions in the response portion.
        2. Upweight a ±4 token window around each '=' at 5x.
           Rationale: these are computation checkpoints where the model
           must produce the arithmetically correct result. Strong signal here
           teaches arithmetic consistency.
        3. Find the anchor (####) and upweight anchor+answer at 2x (not 10x).
           Rationale: format is already learned. We don't need 10x to maintain it.
           Reducing this redirects gradient budget toward computation steps.
        4. Everything else in the response stays at 1x.
        """
        n = len(ids)
        weights = [0.0] * p_ids_len + [1.0] * (n - p_ids_len)

        response_ids = ids[p_ids_len:]

        # ── Step 1: Find '=' positions and upweight computation windows ──
        COMPUTATION_WEIGHT = 5.0
        WINDOW = 4  # tokens on each side of '='

        for pos, token_id in enumerate(response_ids):
            if token_id in self._eq_token_ids:
                # Upweight the window around this '='
                start = max(0, pos - WINDOW)
                end = min(len(response_ids), pos + WINDOW + 1)
                for k in range(start, end):
                    # Take the max in case windows overlap — don't double-count
                    abs_k = p_ids_len + k
                    if abs_k < n:
                        weights[abs_k] = max(weights[abs_k], COMPUTATION_WEIGHT)

        # ── Step 2: Find anchor and apply moderate anchor weight ──
        ANCHOR_WEIGHT = 2.0  # Reduced from 10x to 2x

        anchor_start_in_response = -1
        for j in range(len(response_ids) - self.stable_anchor_len + 1):
            if response_ids[j: j + self.stable_anchor_len] == self.stable_anchor_ids:
                anchor_start_in_response = j
                break

        if anchor_start_in_response != -1:
            # Upweight from anchor start to end of sequence
            for k in range(anchor_start_in_response, len(response_ids)):
                abs_k = p_ids_len + k
                if abs_k < n:
                    # Use max so computation window and anchor don't conflict
                    weights[abs_k] = max(weights[abs_k], ANCHOR_WEIGHT)
        else:
            # Anchor not found (shouldn't happen after filtering) — mild tail weight
            fallback_start = max(p_ids_len, n - 8)
            for k in range(fallback_start, n):
                weights[k] = max(weights[k], ANCHOR_WEIGHT)

        return weights

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, i: int) -> Dict[str, List[int]]:
        rec = self.records[i]
        prompt = PROMPT_TEMPLATE.format(q=rec["query_vi"].strip())
        response = rec.get("_target", rec.get("response_vi", "")).strip()

        p_ids = self.tok(prompt, add_special_tokens=False)["input_ids"]
        r_ids = self.tok(response, add_special_tokens=False)["input_ids"] + [SAFE_EOS_ID]

        ids = (p_ids + r_ids)[: self.max_length]
        labels = ([-100] * len(p_ids) + r_ids)[: self.max_length]

        prompt_end = len(p_ids)
        weights = self._build_arithmetic_weights(ids, prompt_end)

        # Token-id safety clamp
        ids = [min(t, SAFE_EOS_ID) for t in ids]
        labels = [(-100 if t == -100 else min(t, SAFE_EOS_ID)) for t in labels]

        return {
            "input_ids": ids,
            "labels": labels,
            "attention_mask": [1] * len(ids),
            "loss_weights": weights,
        }

In [15]:
@dataclass
class PadCollator:
    pad_id: int = SAFE_EOS_ID

    def __call__(self, batch):
        maxlen = max(len(x["input_ids"]) for x in batch)
        out = {"input_ids": [], "attention_mask": [], "labels": []}
        has_weights = "loss_weights" in batch[0]
        if has_weights:
            out["loss_weights"] = []
            
        for x in batch:
            n = len(x["input_ids"])
            pad = maxlen - n
            out["input_ids"].append(x["input_ids"] + [self.pad_id] * pad)
            out["attention_mask"].append(x["attention_mask"] + [0] * pad)
            out["labels"].append(x["labels"] + [-100] * pad)
            if has_weights:
                # Pad weights với 0 (sẽ bị mask anyway)
                out["loss_weights"].append(x["loss_weights"] + [0.0] * pad)
        
        result = {k: torch.tensor(v, dtype=torch.long) 
                  for k, v in out.items() if k != "loss_weights"}
        if has_weights:
            result["loss_weights"] = torch.tensor(
                out["loss_weights"], dtype=torch.float32
            )
        return result

In [16]:
# ── Weight Verification Cell ──
test_ds = SFTDataset(train_records, tokenizer, MAX_LENGTH)
sample = test_ds[0]
ids = sample["input_ids"]
weights = sample["loss_weights"]

# Decode and show weight distribution
tokens_with_weights = [
    (tokenizer.decode([tid]), w)
    for tid, w in zip(ids, weights)
    if w > 0  # skip prompt (weight=0)
]

print("=== Weight Distribution Check ===")
print(f"Tokens with weight=1.0 (baseline response): "
      f"{sum(1 for _, w in tokens_with_weights if w == 1.0)}")
print(f"Tokens with weight=5.0 (computation windows): "
      f"{sum(1 for _, w in tokens_with_weights if w == 5.0)}")
print(f"Tokens with weight=2.0 (anchor+answer): "
      f"{sum(1 for _, w in tokens_with_weights if w == 2.0)}")

print("\n=== High-Weight Tokens (≥2x) ===")
for tok_str, w in tokens_with_weights:
    if w >= 2.0:
        print(f"  w={w:.1f}  |  '{tok_str}'")

# Confirm anchor still detected
decoded = tokenizer.decode(ids)
assert "####đáp án là:" in decoded, "FAIL: Anchor missing from sample — check normalize_response"
print("\n[PASS] Anchor present. Weight rebalancing applied correctly.")

=== Weight Distribution Check ===
Tokens with weight=1.0 (baseline response): 40
Tokens with weight=5.0 (computation windows): 55
Tokens with weight=2.0 (anchor+answer): 7

=== High-Weight Tokens (≥2x) ===
  w=5.0  |  ' Alex'
  w=5.0  |  ' đang'
  w=5.0  |  ' mời'
  w=5.0  |  ' 2'
  w=5.0  |  '/'
  w=5.0  |  '3'
  w=5.0  |  ' số'
  w=5.0  |  ' khách'
  w=5.0  |  ' đó'
  w=5.0  |  ','
  w=5.0  |  ' tức'
  w=5.0  |  ' là'
  w=5.0  |  ' 84'
  w=5.0  |  ' *'
  w=5.0  |  ' 2'
  w=5.0  |  '/'
  w=5.0  |  '3'
  w=5.0  |  ' ='
  w=5.0  |  ' 56'
  w=5.0  |  ' khách'
  w=5.0  |  '.'
  w=5.0  |  ' Vậy'
  w=5.0  |  ' số'
  w=5.0  |  ' khách'
  w=5.0  |  ' là'
  w=5.0  |  ' 84'
  w=5.0  |  ' +'
  w=5.0  |  ' 56'
  w=5.0  |  ' ='
  w=5.0  |  ' 140'
  w=5.0  |  ' khách'
  w=5.0  |  '.'
  w=5.0  |  ' Người'
  w=5.0  |  ' cần'
  w=5.0  |  ' dùng'
  w=5.0  |  ' là'
  w=5.0  |  ' 140'
  w=5.0  |  ' +'
  w=5.0  |  ' 10'
  w=5.0  |  ' ='
  w=5.0  |  ' 150'
  w=5.0  |  ' đĩa'
  w=5.0  |  '.'
  w=5.0  |  ' M

## Greedy generation

In [ ]:
# ============================================================
# 4. Stabilized Self-Consistency Generation
# ============================================================
from collections import Counter
from torch.utils.data import DataLoader

def extract_numeric_answer_for_voting(text: str) -> str:
    """Tối ưu riêng cho Self-Consistency: Bắt mọi biến thể mỏ neo."""
    match = re.search(r"(?i)(?:####\s*|đáp án là\s*[:：]?\s*)+(-?\d+(?:[.,]\d+)?)", text)
    if match:
        return match.group(1).replace(',', '.')
    numbers = re.findall(r"(-?\d+(?:[.,]\d+)?)", text)
    if numbers:
        return numbers[-1].replace(',', '.')
    return "NONE"

def generate_outputs(
    model, tokenizer, records: list, 
    batch_size: int = 8,        # Tối ưu hóa GPU T4 (Có thể lên 16 nếu VRAM cho phép)
    max_new_tokens: int = 150,
    num_samples: int = 8,       # 3 rollouts là điểm ngoạt an toàn cho Kaggle
    temperature: float = 0.4,
    device: str = "cuda"
) -> list:
    
    tokenizer.padding_side = "left"
    tokenizer.pad_token_id = SAFE_EOS_ID
    
    outputs = []
    
    # Chuyển records thành DataLoader để xử lý theo batch
    for i in tqdm(range(0, len(records), batch_size), desc="Batched Self-Consistency"):
        batch_records = records[i:i + batch_size]
        prompts = [PROMPT_TEMPLATE.format(q=r["query_vi"].strip()) for r in batch_records]
        
        enc = tokenizer(
            prompts, return_tensors="pt", truncation=True, padding=True,
            max_length=MAX_LENGTH, add_special_tokens=False
        ).to(device)
        
        ids = enc["input_ids"].clamp(max=model.config.vocab_size - 1)
        attention_mask = enc["attention_mask"]

        with torch.inference_mode():
            gen = model.generate(
                input_ids=ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_k=30,
                num_return_sequences=num_samples,
                repetition_penalty=1.05,  # FIXED: Cho phép toán học lặp lại tự nhiên
                pad_token_id=SAFE_EOS_ID, 
                eos_token_id=SAFE_EOS_ID,
            )
        
        prompt_len = ids.shape[1]
        
        for j, rec in enumerate(batch_records):
            # Cắt lấy (num_samples) kết quả thuộc về record hiện tại
            start_idx = j * num_samples
            end_idx = start_idx + num_samples
            candidates_tokens = gen[start_idx:end_idx, prompt_len:]
            
            candidates = tokenizer.batch_decode(candidates_tokens, skip_special_tokens=True)
            
            # --- Logic Voting Toán học ---
            parsed = []
            for c in candidates:
                ans_str = extract_numeric_answer_for_voting(c)
                if ans_str != "NONE":
                    num = parse_number(ans_str)
                    if num is not None:
                        parsed.append((num, c))
            
            if not parsed:
                best_output = sorted(candidates, key=len)[0]
            else:
                counter = Counter(round(n, 4) for n, _ in parsed)
                consensus = counter.most_common(1)[0][0]
                consensus_texts = [c for n, c in parsed if round(n, 4) == consensus]
                best_output = sorted(consensus_texts, key=len)[0] # Anti-hallucination tie-breaker
            
            outputs.append({
                "id": rec.get("id", len(outputs)),
                "query_vi": rec["query_vi"],
                "type": rec.get("type"),
                "model_output": best_output,
            })
            
    return outputs

## Training

In [20]:
# ============================================================
# 7. Train (WITH CUSTOM WEIGHTED TRAINER)
# ============================================================
import torch.nn.functional as F
from torch.utils.data import DataLoader, SequentialSampler

class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        loss_weights = inputs.pop("loss_weights", None)
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss_fct     = torch.nn.CrossEntropyLoss(reduction="none")
        
        per_token_loss = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1)
        ).view(shift_labels.shape)
        
        mask = (shift_labels != -100).float()
        
        if loss_weights is not None:
            shift_weights = loss_weights[:, 1:].contiguous().to(per_token_loss.device)
            per_token_loss = per_token_loss * shift_weights
            # FIX: Normalize by the sum of the active weights, not just the mask
            weight_sum = (mask * shift_weights).sum().clamp(min=1.0)
            weighted_loss = (per_token_loss * mask).sum() / weight_sum
        else:
            weighted_loss = (per_token_loss * mask).sum() / mask.sum().clamp(min=1.0)
            
        return (weighted_loss, outputs) if return_outputs else weighted_loss
        
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    local_files_only=True
)
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    local_files_only=True
)
model.config.pad_token_id = SAFE_EOS_ID
model.config.eos_token_id = SAFE_EOS_ID


model.gradient_checkpointing_enable()
model.config.use_cache = False

train_ds = SFTDataset(train_records, tokenizer, MAX_LENGTH)
valid_ds = SFTDataset(valid_records, tokenizer, MAX_LENGTH)
collator = PadCollator(pad_id=SAFE_EOS_ID)

eff_batch = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM * max(1, torch.cuda.device_count())
steps_per_epoch = math.ceil(len(train_ds) / eff_batch)
print(
    f"per_device_bs={PER_DEVICE_BATCH_SIZE} | grad_accum={GRAD_ACCUM} | "
    f"gpus={torch.cuda.device_count()} | effective_batch={eff_batch} | "
    f"steps/epoch={steps_per_epoch}"
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


per_device_bs=4 | grad_accum=4 | gpus=2 | effective_batch=32 | steps/epoch=1619


In [21]:
# VERIFICATION TEST: Ensure Loss Weighting hit the target
sample = train_ds[0]
weights = sample["loss_weights"]
ids = sample["input_ids"]

upweighted_indices = [i for i, w in enumerate(weights) if w > 1.0]
assert len(upweighted_indices) > 0, "FATAL: Anchor matching failed. Weights are 1.0 everywhere."

# Print what is actually being upweighted to ensure it caught the numbers
print("Upweighted text:", tokenizer.decode([ids[i] for i in upweighted_indices]))

Upweighted text:  Alex đang mời 2/3 số khách đó, tức là 84 * 2/3 = 56 khách. Vậy số khách là 84 + 56 = 140 khách. Người cần dùng là 140 + 10 = 150 đĩa. Mỗi cần thiết là 150 * 8 = 1200 ngọn măng tây####đáp án là: 1200hue


In [22]:
ta_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    weight_decay=WEIGHT_DECAY,
    fp16=torch.cuda.is_available(),
    optim="adamw_torch_fused",                   # ADDED: Faster execution on T4
    logging_steps=20,
    logging_strategy="steps",        # Bắt buộc khai báo chiến lược logging
    log_level="info",                # Đảm bảo mức độ log đủ thấp để hiển thị
    report_to="none",                # Giữ nguyên nếu bạn không muốn dùng WandB
    save_strategy="epoch",
    save_total_limit=1,
    seed=SEED,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,                  # ADDED: Speeds up host-to-device transfer
    remove_unused_columns=False,
)

sig = inspect.signature(TrainingArguments.__init__)
if "eval_strategy" in sig.parameters:
    ta_kwargs["eval_strategy"] = "epoch"
else:
    ta_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**ta_kwargs)

# Use the custom trainer
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=collator,
)

t0 = time.time()
trainer.train()
train_dt = time.time() - t0
print(f"\n[train] wall time: {train_dt:.1f}s ({train_dt/60:.2f} min)")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

model_hash = sha256_dir(OUTPUT_DIR)
with (OUTPUT_DIR / "model_hash.txt").open("w", encoding="utf-8") as f:
    f.write(model_hash + "\n")

print("Saved checkpoint to:", OUTPUT_DIR)
print("Model SHA256:", model_hash)

# Free training objects before inference reload.
del trainer, model
torch.cuda.empty_cache()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
***** Running training *****
  Num examples = 51,787
  Num Epochs = 3
  Instantaneous batch size per device = 4
  Training with DataParallel so batch size has been adjusted to: 8
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 4
  Total optimization steps = 4,857
  Number of trainable parameters = 124,439,808


Epoch,Training Loss,Validation Loss
1,4.642453,1.487556
2,4.135576,1.404102
3,3.937675,1.399398



***** Running Evaluation *****
  Num examples = 500
  Batch size = 8
Saving model checkpoint to /kaggle/working/gpt2_math_ckpt/checkpoint-1619
Configuration saved in /kaggle/working/gpt2_math_ckpt/checkpoint-1619/config.json
Configuration saved in /kaggle/working/gpt2_math_ckpt/checkpoint-1619/generation_config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in /kaggle/working/gpt2_math_ckpt/checkpoint-1619/model.safetensors
Deleting older checkpoint [/kaggle/working/gpt2_math_ckpt/checkpoint-3728] due to args.save_total_limit

***** Running Evaluation *****
  Num examples = 500
  Batch size = 8
Saving model checkpoint to /kaggle/working/gpt2_math_ckpt/checkpoint-3238
Configuration saved in /kaggle/working/gpt2_math_ckpt/checkpoint-3238/config.json
Configuration saved in /kaggle/working/gpt2_math_ckpt/checkpoint-3238/generation_config.json


...

Model weights saved in /kaggle/working/gpt2_math_ckpt/checkpoint-4857/model.safetensors
Deleting older checkpoint [/kaggle/working/gpt2_math_ckpt/checkpoint-3238] due to args.save_total_limit


Training completed. Do not forget to share your model on huggingface.co/models =)


Saving model checkpoint to /kaggle/working/gpt2_math_ckpt
Configuration saved in /kaggle/working/gpt2_math_ckpt/config.json
Configuration saved in /kaggle/working/gpt2_math_ckpt/generation_config.json



[train] wall time: 8364.8s (139.41 min)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in /kaggle/working/gpt2_math_ckpt/model.safetensors
tokenizer config file saved in /kaggle/working/gpt2_math_ckpt/tokenizer_config.json


Saved checkpoint to: /kaggle/working/gpt2_math_ckpt
Model SHA256: 435bf01c866aee03da495ca46a26cacd42c6600814273f338376c573e181e938


In [56]:
# ======================================================
# QUICK DIAGNOSTIC
# ======================================================
valid_records_gsm = [r for r in valid_records if r.get('type', '').startswith('GSM')]
valid_records_math = [r for r in valid_records if r.get('type', '').startswith('MATH')]
print(f"Val set: {len(valid_records_gsm)} GSM, {len(valid_records_math)} MATH")

EPOCH1_CKPT = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    torch_dtype=torch.float16,
    device_map="auto"
)

quick_valid = valid_records_gsm[:20]  # Chỉ GSM để test nhanh

quick_outputs = generate_outputs(
    model=EPOCH1_CKPT,
    tokenizer=tokenizer,
    records=quick_valid,
    max_new_tokens=150
)

# Save outputs
with open("/kaggle/working/quick_check.json", "w", encoding="utf-8") as f:
    json.dump(quick_outputs, f, ensure_ascii=False, indent=2)

# Reload predictions
with open("/kaggle/working/quick_check.json", "r", encoding="utf-8") as f:
    preds = json.load(f) 

# 2. Corrected Quick Check Loop
correct = 0
for pred, gold in zip(preds, quick_valid):
    pred_ans = extract_pred(pred)
    gold_ans = extract_gold(gold)  
    
    pred_num = parse_number(pred_ans)
    gold_num = parse_number(gold_ans)
    
    # CRITICAL FIX: Explicit None check protects zero-value answers
    if pred_num is not None and gold_num is not None: 
        err = abs(pred_num - gold_num) / max(1.0, abs(gold_num))
        if err <= 0.01:
            correct += 1
            print(f"CORRECT: pred={pred_num}, gold={gold_num}")
        else:
            print(f"WRONG: pred={pred_num}, gold={gold_num}, err={err:.2f}")
    else:
        # Improved logging to show BOTH variables
        print(f"PARSE FAILED: pred_ans='{pred_ans}' -> {pred_num} | gold_ans='{gold_ans}' -> {gold_num}")

print(f"\nQuick score: {correct}/20 = {correct/20*100:.0f}%")

loading configuration file /kaggle/working/gpt2_math_ckpt/config.json
Model config GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.0,
  "bos_token_id": 50256,
  "dtype": "float16",
  "embd_pdrop": 0.0,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": 50256,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.0,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "tie_word_embeddings": true,
  "transformers_version":

Val set: 312 GSM, 188 MATH


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

loading configuration file /kaggle/working/gpt2_math_ckpt/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 50256,
  "eos_token_id": 50256,
  "use_cache": true
}



Batched Self-Consistency:   0%|          | 0/3 [00:00<?, ?it/s]

WRONG: pred=108.0, gold=37.0, err=1.92
WRONG: pred=30.0, gold=39.0, err=0.23
WRONG: pred=640.0, gold=440.0, err=0.45
WRONG: pred=300.0, gold=90.0, err=2.33
WRONG: pred=108.0, gold=12.0, err=8.00
WRONG: pred=8.0, gold=9.0, err=0.11
WRONG: pred=3600.0, gold=5.0, err=719.00
WRONG: pred=160.0, gold=1.0, err=159.00
WRONG: pred=13.0, gold=3.0, err=3.33
WRONG: pred=64.0, gold=80.0, err=0.20
WRONG: pred=3.5, gold=18.0, err=0.81
WRONG: pred=100.0, gold=460.0, err=0.78
WRONG: pred=40.0, gold=10.0, err=3.00
WRONG: pred=198.0, gold=477.0, err=0.58
WRONG: pred=47.0, gold=11.0, err=3.27
WRONG: pred=4.0, gold=25.0, err=0.84
WRONG: pred=10.0, gold=2.0, err=4.00
WRONG: pred=180.0, gold=360.0, err=0.50
CORRECT: pred=10.0, gold=10.0
WRONG: pred=22.0, gold=45.0, err=0.51

Quick score: 1/20 = 5%


## Inference on validation set

In [50]:
# ============================================================
# 8. Inference on validation set
# ============================================================
# valid_outputs = generate_outputs(OUTPUT_DIR, valid_records, VALID_OUTPUT_PATH, max_new_tokens=MAX_NEW_TOKENS)

valid_outputs = generate_outputs(
    model=EPOCH1_CKPT,
    tokenizer=tokenizer,
    records=valid_records,
    max_new_tokens=150
)
print("\nExample output:")
print(json.dumps(valid_outputs[0], ensure_ascii=False, indent=2)[:2000])


Batched Self-Consistency:   0%|          | 0/63 [00:00<?, ?it/s]


Example output:
{
  "id": 0,
  "query_vi": "Nếu Susan đang chơi 1 trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về phía trước 8 ô, ở lượt thứ hai, cô ấy di chuyển 2 ô nhưng bị đẩy lùi lại 5 ô và ở lượt thứ ba. đến lượt cô ấy tiến về phía trước 6 ô, cô ấy cần di chuyển thêm bao nhiêu ô nữa để đến ô cuối và giành chiến thắng trong trò chơi?",
  "type": "GSM_Rephrased",
  "model_output": "Susan đã tiến hành 2 ô bắt đầu và 4 ô cuối cùng, tổng + là 2 + 4 = 6 ô. Cô ấy cần di chuyển thêm 5 ô nữa nên cô ấy cần di chuyển thêm 6 - 5 = 3 ô nữa. Để tìm số ô còn lại, chúng ta cộng số ô rồi chia cho số ô bắt đầu, sẽ được 3^2=7 ô. Vì vậy, Susan cần di chuyển thêm 3 ô nữa để đến ô cuối cùng.\n####đáp án là: 7huehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehue"
}


## Evaluate

In [51]:
# ============================================================
# 9. Evaluate
# ============================================================
with open(VALID_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(valid_outputs, f, ensure_ascii=False, indent=2)

print("\nExample output:")
print(json.dumps(valid_outputs[0], ensure_ascii=False, indent=2)[:2000])

# Evaluate
valid_result = save_evaluation_report(
    VALID_OUTPUT_PATH,
    valid_records,
    VALID_REPORT_PATH,
)

summary = valid_result["summary"]

print("\nFinal validation score:")
print(f'{summary["raw_score"]} / {summary["max_raw_score"]}  ({summary["score_pct"]*100:.2f}%)')
print(f'Score /10: {summary["score_10"]:.2f}')
print("Buckets:", summary["buckets"])

# Create run log for experiment tracking
run_log = {
    "run_id": "p1_canonical",
    "data_format": "canonical_full",
    "train_samples": len(train_records),
    "lr": LR, 
    "max_length": MAX_LENGTH, 
    "max_new_tokens": MAX_NEW_TOKENS,
    "epochs": EPOCHS, 
    "decoding": "greedy_antirep_1.15",
    "score_10": summary["score_10"], 
    "raw_score": summary["raw_score"], 
    "extractable": summary["extractable"],
    "buckets": summary["buckets"], 
    "runtime_train_min": round(train_dt / 60, 2) if 'train_dt' in locals() else None, 
    "score_by_type": None # Optional: Compute if you do type-specific groupby later
}

print("\n--- EXPERIMENT LOG ---")
print(json.dumps(run_log, indent=2))


Example output:
{
  "id": 0,
  "query_vi": "Nếu Susan đang chơi 1 trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về phía trước 8 ô, ở lượt thứ hai, cô ấy di chuyển 2 ô nhưng bị đẩy lùi lại 5 ô và ở lượt thứ ba. đến lượt cô ấy tiến về phía trước 6 ô, cô ấy cần di chuyển thêm bao nhiêu ô nữa để đến ô cuối và giành chiến thắng trong trò chơi?",
  "type": "GSM_Rephrased",
  "model_output": "Susan đã tiến hành 2 ô bắt đầu và 4 ô cuối cùng, tổng + là 2 + 4 = 6 ô. Cô ấy cần di chuyển thêm 5 ô nữa nên cô ấy cần di chuyển thêm 6 - 5 = 3 ô nữa. Để tìm số ô còn lại, chúng ta cộng số ô rồi chia cho số ô bắt đầu, sẽ được 3^2=7 ô. Vì vậy, Susan cần di chuyển thêm 3 ô nữa để đến ô cuối cùng.\n####đáp án là: 7huehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehue"
}
{
  "n": 500,
  "raw_score": 491,
  "max_raw_score": 5000,
  "score_10": 0.982,
  "score_pct": 0.0982,
 

In [52]:
# ============================================================
# 10. Detailed Error analysis
# ============================================================
rows = valid_result["rows"]
bad = [(i, r) for i, r in enumerate(rows) if r.get("score", 0) == 0]
good = [(i, r) for i, r in enumerate(rows) if r.get("score", 0) == 10]

print("Good:", len(good), "| Bad:", len(bad))

for i, r in bad[:2]:
    pred = valid_outputs[i]
    gold = valid_records[i]
    print("=" * 100)
    print("IDX:", i, "| type:", gold.get("type"), "| rel_error:", r.get("rel_error"))
    print("QUERY:", gold["query_vi"][:500])
    print("\nGOLD:", gold["response_vi"][-500:])
    print("\nPRED:", pred["model_output"][:1000])


Good: 35 | Bad: 352
IDX: 0 | type: GSM_Rephrased | rel_error: 0.8108108108108109
QUERY: Nếu Susan đang chơi 1 trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về phía trước 8 ô, ở lượt thứ hai, cô ấy di chuyển 2 ô nhưng bị đẩy lùi lại 5 ô và ở lượt thứ ba. đến lượt cô ấy tiến về phía trước 6 ô, cô ấy cần di chuyển thêm bao nhiêu ô nữa để đến ô cuối và giành chiến thắng trong trò chơi?

GOLD: Ở lượt đầu tiên, Susan tiến về phía trước 8 ô, vì vậy cô ấy hiện ở ô 8. Ở lượt thứ hai, cô ấy tiến về phía trước 2 ô, nhưng bị đẩy lùi 5 ô, vì vậy cô ấy kết thúc ở ô 8 + 2 - 5 = 5 Ở lượt thứ 3, cô ấy tiến về phía trước 6 ô nên lúc này cô ấy đang ở ô 5 + 6 = 11. Ô cuối cùng thắng là 48 nên Susan cần di chuyển thêm 48 - 11 = 37 ô nữa để đến ô cuối cùng và giành chiến thắng tro choi. #### 37 Đáp án là: 37

PRED: Susan đã tiến hành 2 ô bắt đầu và 4 ô cuối cùng, tổng + là 2 + 4 = 6 ô. Cô ấy cần di chuyển thêm 5 ô nữa nên cô ấy cần di chuyển thêm 6 - 5 = 3 ô nữa.

In [53]:
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict

class VietnameseMathErrorAnalyzer:
    def __init__(self, valid_report_path: str | Path, valid_output_path: str | Path, tokenizer=None):
        self.report_path = Path(valid_report_path)
        self.output_path = Path(valid_output_path)
        self.tokenizer = tokenizer
        
        # Load datasets
        with self.report_path.open("r", encoding="utf-8") as f:
            self.report_data = json.load(f)
        with self.output_path.open("r", encoding="utf-8") as f:
            self.output_data = json.load(f)
            
        self.rows = self.report_data["rows"]
        self.summary = self.report_data["summary"]
        
        # Build faster lookup mapping
        self.pred_map = {i: x for i, x in enumerate(self.output_data)}
        self.df = None

    # ============================================================
    # 1. Failure Taxonomy & GPT2 Degeneration Core
    # ============================================================
    def _classify_failure_modes(self, row: dict, pred_item: dict) -> list[str]:
        """Heuristic rule engine classifying classic decoder-only math degradation signatures."""
        modes = []
        output_text = pred_item.get("model_output", "")
        gold_text = row.get("gold_answer", "") or ""
        
        # Handle cases where answer tracking fails completely
        if not row["extractable"]:
            modes.append("extraction_failure")
            
        # Detect Premature End of Sequence (EOS) vs Verbosity Truncation
        approx_words = len(output_text.split())
        if approx_words <= 4 and row["score"] == 0:
            modes.append("premature_eos")
        elif approx_words >= 195:  # Close to the 200 token MAX_NEW_TOKENS boundary
            modes.append("truncation_failure")

        # Repetition Loop Detectors (Classic GPT-2 Signature)
        # Check for repeated N-grams or continuous phrases
        words = output_text.lower().split()
        if len(words) > 10:
            trigrams = [" ".join(words[i:i+3]) for i in range(len(words)-2)]
            if len(trigrams) > 0:
                most_common_tri = Counter(trigrams).most_common(1)[0]
                if most_common_tri[1] > 4:  # Trigram repeats more than 4 times
                    modes.append("repetition_loop")

        # Check for numeric anomalies / formatting drift
        # Check for numeric anomalies / formatting drift
        if row["extractable"] and row["score"] == 0:
            pred_num = row["pred_num"]
            gold_num = row["gold_num"]
            
            if pred_num is not None and gold_num is not None:
                # Sign Mistakes
                if pred_num == -gold_num and gold_num != 0:
                    modes.append("operator_sign_mistake")
                # Decimal placement shifts
                elif gold_num != 0 and pred_num != 0:
                    ratio = abs(pred_num / gold_num)
                    # Guard against float overflow/underflow resulting in inf or 0
                    if ratio > 0 and not np.isinf(ratio):
                        log_ratio = np.log10(ratio)
                        # Check if log_ratio is extremely close to an integer (e.g., 10^1, 10^-2)
                        if abs(log_ratio - round(log_ratio)) < 1e-4:
                            modes.append("decimal_placement_error")
                        else:
                            modes.append("arithmetic_mistake")
                    else:
                        modes.append("arithmetic_mistake")
                else:
                    # Catch-all for when one value is 0 but they don't match
                    modes.append("arithmetic_mistake")
            else:
                modes.append("formatting_drift")
                
        # Bilingual Leakage Heuristics
        en_words = re.findall(r"\b(the|answer|is|let|hence|therefore|step|solution)\b", output_text, re.I)
        if len(en_words) >= 2:
            modes.append("bilingual_leakage")
            
        if not modes and row["score"] < 10:
            modes.append("reasoning_collapse")
            
        return modes if modes else ["clean_pass"]

    # ============================================================
    # 2. Pipeline Execution Engine
    # ============================================================
    def build_diagnostic_dataframe(self):
        """Assembles comprehensive failure profiling metrics across columns."""
        processed_rows = []
        
        for idx, r in enumerate(self.rows):
            p = self.pred_map[idx]
            out_text = p.get("model_output", "")
            
            # Compute character lengths & math structural density
            pred_len_chars = len(out_text)
            pred_words = out_text.split()
            math_ops_count = len(re.findall(r"[\+\-\*\/=]|\\frac", out_text))
            math_density = math_ops_count / max(1, len(pred_words))
            
            # Identify precise taxonomy classes
            failure_categories = self._classify_failure_modes(r, p)
            
            # Compute raw tokenizer sequence lengths if optional tokenizer available
            token_count = len(self.tokenizer.encode(out_text)) if self.tokenizer else len(pred_words) * 1.3
            
            processed_rows.append({
                "id": r["id"],
                "type": r["type"],
                "score": r["score"],
                "extractable": int(r["extractable"]),
                "rel_error": r["rel_error"] if r["rel_error"] is not None else np.nan,
                "pred_len_tokens": token_count,
                "math_density": math_density,
                "failures": failure_categories,
                "primary_failure": failure_categories[0]
            })
            
        self.df = pd.DataFrame(processed_rows)
        return self.df

    # ============================================================
    # 3. Analytics & Score-Bucket Sweeper
    # ============================================================
    def generate_dashboard_metrics(self) -> dict:
        """Computes structural aggregations mapping failure classes directly to Score loss."""
        if self.df is None:
            self.build_diagnostic_dataframe()
            
        metrics = {}
        
        # Explode failures list to get true distribution counts
        exploded_df = self.df.explode("failures")
        failure_counts = exploded_df["failures"].value_counts()
        avg_score_by_failure = exploded_df.groupby("failures")["score"].mean()
        
        # Assemble Primary Damage Inventory Matrix
        damage_matrix = pd.DataFrame({
            "Occurrences": failure_counts,
            "Mean_Score_Impact": avg_score_by_failure
        }).sort_values(by="Occurrences", ascending=False)
        
        metrics["failure_damage_table"] = damage_matrix
        
        # Analyze performance across math problem tracks
        metrics["type_performance"] = self.df.groupby("type")["score"].agg(["count", "mean"]).to_dict(orient="index")
        
        # Metric Extraction Diagnostics Dashboard
        metrics["extraction_leakage_rate"] = 1.0 - (self.df["extractable"].sum() / len(self.df))
        
        return metrics

    # ============================================================
    # 4. Simulation Engine: Extraction Fallbacks (No Retraining ROI)
    # ============================================================
    def simulate_fallback_extractor_gain(self) -> float:
        """Simulates potential validation point recovery using post-hoc regex without retraining."""
        simulated_score = 0
        recovered_points = 0
        
        for idx, r in enumerate(self.rows):
            p = self.pred_map[idx]
            out_text = p.get("model_output", "")
            
            if r["score"] > 0:
                simulated_score += r["score"]
                continue
                
            # Simulate an aggressive reverse number matching scraper
            nums = re.findall(r"-?\d+(?:[,\.]\d+)?", out_text)
            if nums:
                fallback_pred_str = nums[-1].replace(",", ".")
                try:
                    fallback_pred = float(fallback_pred_str)
                    gold_num = float(r["gold_num"]) if r["gold_num"] is not None else None
                    
                    if gold_num is not None:
                        denom = max(1.0, abs(gold_num))
                        error = abs(fallback_pred - gold_num) / denom
                        
                        # Calculate recovery tier
                        this_score = 0
                        if error <= 0.01: this_score = 10
                        elif error <= 0.10: this_score = 5
                        elif error <= 0.50: this_score = 1
                        
                        recovered_points += this_score
                        simulated_score += this_score
                except ValueError:
                    pass
                    
        baseline_score = self.summary["raw_score"]
        max_possible = len(self.rows) * 10
        print(f"\n[EXTRACTION SIMULATOR] Baseline Validation Score: {baseline_score}/{max_possible}")
        print(f"[EXTRACTION SIMULATOR] Simulated Fallback Score: {baseline_score + recovered_points}/{max_possible}")
        print(f"[EXTRACTION SIMULATOR] Net Recovery Delta: +{recovered_points} points ({((recovered_points)/max_possible)*100:.2f}%)")
        return recovered_points

    # ============================================================
    # 5. Formatted Markdown Reporting Tool
    # ============================================================
    def print_terminal_markdown_report(self):
        """Generates clear, scannable terminal outputs for experiment tracking."""
        metrics = self.generate_dashboard_metrics()
        
        print("\n" + "="*80)
        print("         COMPETITION DATA-CENTRIC ERROR DIAGNOSTIC REPORT")
        print("="*80)
        
        print(f"\n### SUMMARY PROFILE")
        print(f"* Raw Base Validation Score: {self.summary['raw_score']} / {self.summary['max_raw_score']} ({self.summary['score_pct']*100:.2f}%)")
        print(f"* Total Dataset Samples Analyzed: {self.summary['n']}")
        print(f"* Unextractable Pipeline Leakage: {metrics['extraction_leakage_rate']*100:.2f}%")
        
        print("\n### FAILURE MATRIX DAMAGE DISTRIBUTION")
        print(metrics["failure_damage_table"].to_markdown())
        
        print("\n### SUB-TRACK TRACKING PERFORMANCE")
        track_df = pd.DataFrame.from_dict(metrics["type_performance"], orient="index")
        print(track_df.to_markdown())
        
        self.simulate_fallback_extractor_gain()
        print("\n" + "="*80)

In [54]:
analyzer = VietnameseMathErrorAnalyzer(VALID_REPORT_PATH, VALID_OUTPUT_PATH, tokenizer=tokenizer)
analyzer.print_terminal_markdown_report()


         COMPETITION DATA-CENTRIC ERROR DIAGNOSTIC REPORT

### SUMMARY PROFILE
* Raw Base Validation Score: 491 / 5000 (9.82%)
* Total Dataset Samples Analyzed: 500
* Unextractable Pipeline Leakage: 1.60%

### FAILURE MATRIX DAMAGE DISTRIBUTION
| failures                |   Occurrences |   Mean_Score_Impact |
|:------------------------|--------------:|--------------------:|
| arithmetic_mistake      |           332 |             0       |
| reasoning_collapse      |            97 |             1.28866 |
| repetition_loop         |            92 |             1.15217 |
| clean_pass              |            26 |            10       |
| decimal_placement_error |             8 |             0       |
| extraction_failure      |             8 |             0       |
| formatting_drift        |             4 |             0       |

### SUB-TRACK TRACKING PERFORMANCE
|                |   count |     mean |
|:---------------|--------:|---------:|
| GSM_AnsAug     |     105 | 0.647619 |
| GS

In [55]:
# ============================================================
# 11. List saved artifacts
# ============================================================
for p in [
    VALID_OUTPUT_PATH,
    VALID_REPORT_PATH,
    Path(str(VALID_OUTPUT_PATH) + ".sha256.txt"),
    OUTPUT_DIR / "model_hash.txt",
]:
    print(p, "exists=", p.exists(), "size=", p.stat().st_size if p.exists() else None)

print("\nKaggle output folder:")
for p in sorted(Path("/kaggle/working").glob("*")):
    print("-", p)


/kaggle/working/valid_output.json exists= True size= 434102
/kaggle/working/valid_report.json exists= True size= 118371
/kaggle/working/valid_output.json.sha256.txt exists= True size= 65
/kaggle/working/gpt2_math_ckpt/model_hash.txt exists= True size= 65

Kaggle output folder:
- /kaggle/working/.virtual_documents
- /kaggle/working/gpt2_math_ckpt
- /kaggle/working/quick_check.json
- /kaggle/working/quick_check.json.sha256.txt
- /kaggle/working/valid_output.json
- /kaggle/working/valid_output.json.sha256.txt
- /kaggle/working/valid_report.json
